# 03. Mobility-informed SIR and R(t) estimation for the 2016-2017 season

This notebook estimates time-varying transmission dynamics and the instantaneous reproduction number \(R(t)\) for the 2016-2017 season. The mobility input column `gamma` corresponds to manuscript \(\xi(t)\), while the SIR recovery rate is represented as `sigma = 1.0 / 4.1`. No `theta` parameter is used.


In [ ]:
# Repository path setup
# This cell makes the notebook runnable from either the repository root or the notebooks/ directory.
from pathlib import Path
import os


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate
    if current.name == "notebooks":
        return current.parent
    return current


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

for directory in [
    "data/metro",
    "data/NHIS/2016~2017",
    "data/mobility_factor/2016~2017",
    "data/Rt/2016~2017",
    "data/Rt/2017~2018",
    "data/Rt/2018~2019",
    "data/Rt/2022~2023",
    "figures/mobility_factor",
    "figures/2016~2017",
    "figures/HeatMap",
    "figures/validation",
    "results/validation",
    "results/tables",
]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# 대한민국

from scipy.stats import poisson
from scipy.stats import nbinom
import numpy as np
import matplotlib.pyplot as pt
import time
import pandas as pd
import matplotlib.dates as mdates
from pandas.plotting import register_matplotlib_converters


scale = 1
window = 7
Ntot = 51_737_380
dt = 1.0
cases = 1_000_000
sigma = 1.0 / 4.1
beta_s = 0.15  # particle filtering

# xi
xi = 1

def moving_average(x, w):
    x = np.asarray(x, dtype=float)
    kernel = np.ones(w)

    # 길이는 그대로 유지
    num = np.convolve(x, kernel, mode='same')
    den = np.convolve(np.ones(len(x)), kernel, mode='same')

    return num / den

# Runge-Kutta 4th order
def RK4(S, I, R, beta, xi):
    kS1 = - xi * beta * S * I / Ntot
    kI1 = (xi * beta * S * I / Ntot) - sigma * I
    kR1 = sigma * I

    kS2 = - xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot
    kI2 = (xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot) - sigma * (I + 0.5 * kI1 * dt)
    kR2 = sigma * (I + 0.5 * kI1 * dt)

    kS3 = - xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot
    kI3 = (xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot) - sigma * (I + 0.5 * kI2 * dt)
    kR3 = sigma * (I + 0.5 * kI2 * dt)

    kS4 = - xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot
    kI4 = (xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot) - sigma * (I + kI3 * dt)
    kR4 = sigma * (I + kI3 * dt)

    S = S + (kS1 + 2.0 * kS2 + 2.0 * kS3 + kS4) / 6.0
    I = I + (kI1 + 2.0 * kI2 + 2.0 * kI3 + kI4) / 6.0
    R = R + (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0
    confirm = (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0

    return np.array([S, I, R, confirm])

def transform_datetype(df):
    df['date'] = pd.to_datetime(df['date'])
    return df

np.random.seed()
start = time.time()

# CSV 읽기
data = pd.read_csv("data/NHIS/2016~2017/Korea_cases(2016~2017).csv")
data = transform_datetype(data)

# cases 열만 숫자로 변환
data['cases'] = pd.to_numeric(data['cases'], errors='coerce')

# 숫자로 변환 안 되는 값 제거
data = data.dropna(subset=['cases']).reset_index(drop=True)

# 원본 cases 배열
cases_array = data['cases'].to_numpy(dtype=float)

# moving average 적용
Korea_cases = moving_average(cases_array, window)

# Particle filter Rt
day = len(Korea_cases)

dates = pd.to_datetime(data['date'])
plot_date = dates[0:]

# Initial condition and beta
random_numbers = np.random.normal(0, beta_s, cases * day)
normal_beta = np.reshape(random_numbers, (day, cases))

Pf_S = np.zeros([day + 1, cases])
Pf_I = np.zeros([day + 1, cases])
Pf_R = np.zeros([day + 1, cases])
Pf_beta = np.zeros([day + 1, cases])

rv_index = np.zeros([day + 1, cases], dtype=int)
Pf_confirm = np.zeros([day, cases])
confirm = np.zeros(day)
Pf_RPN = np.zeros(day)
tm = np.zeros(day)

lambda_i = Korea_cases[0]

Pf_I[0, :] = np.random.poisson(lambda_i, size=cases)
Pf_S[0, :] = Ntot - Pf_I[0, :] - Pf_R[0, :]
Pf_beta[0, :] = 1.05 * np.exp(normal_beta[0, :]) * sigma * Ntot / Pf_S[0, :]

n = 40  # dispersion parameter (Negative Binomial distribution)

# time evolution
for k in range(day):
    # print("Calculation:", k)
    tm[k] = k

    Pf_beta[k + 1, :] = Pf_beta[k, :] * np.exp(normal_beta[k, :])
    Pf_results = RK4(Pf_S[k, :], Pf_I[k, :], Pf_R[k, :], Pf_beta[k + 1, :], xi)

    Pf_S[k + 1, :] = Pf_results[0]
    Pf_I[k + 1, :] = Pf_results[1]
    Pf_R[k + 1, :] = Pf_results[2]
    Pf_confirm[k, :] = Pf_results[3]

    # Negative Binomial weight
    weight = nbinom.pmf(round(Korea_cases[k]), n, n / (Pf_confirm[k, :] + n))

    # weight 합이 0이 되는 경우 방지
    weight_sum = np.sum(weight)
    if weight_sum == 0 or not np.isfinite(weight_sum):
        weight = np.ones(cases) / cases
    else:
        weight = weight / weight_sum

    randomList = np.random.choice(np.arange(cases), size=cases, replace=True, p=weight)
    rv_index[k + 1, :] = randomList

    Pf_S[k + 1, :] = Pf_S[k + 1, randomList]
    Pf_I[k + 1, :] = Pf_I[k + 1, randomList]
    Pf_R[k + 1, :] = Pf_R[k + 1, randomList]
    Pf_beta[k + 1, :] = Pf_beta[k + 1, randomList]

Pf_RPN = xi * (np.mean(Pf_beta[1:, :], axis=1) / sigma) / (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Korea_confirm = np.mean(Pf_confirm, axis=1)

# Particle smoother Rt
Ps_S = np.zeros([day + 1, cases])
Ps_beta = np.zeros([day + 1, cases])
Ps_RPN = np.zeros(day)

# t = day
Ps_Pf = np.arange(cases)
Ps_beta[day, :] = Pf_beta[day, Ps_Pf]

# t = day - 1
Ps_Pf = rv_index[day, Ps_Pf]
Ps_beta[day - 1, :] = Pf_beta[day - 1, Ps_Pf]

for k in range(day - 2, -1, -1):
    Ps_Pf = rv_index[k + 1, Ps_Pf]
    Ps_beta[k, :] = Pf_beta[k, Ps_Pf]

Korea_Rt = xi * (np.mean(Ps_beta[1:, :], axis=1) / sigma) / (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Korea_beta = np.mean(Ps_beta[1:, :], axis=1)
# Korea_beta_t = pd.DataFrame({'date': dates[0:],'beta': Korea_beta[0:]})
# Korea_beta_t.to_csv("data/Rt/2016~2017/Korea_beta_0.5(2016~2017).csv")

# Rt Plot
pt.plot(plot_date[10:], Korea_Rt[10:], linestyle='-', linewidth=1.5, color='#000000',  label='$R_t$')
pt.title('Korea Rt')
pt.legend(loc=0)
pt.xlabel('Date')
pt.yticks([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
pt.show()

# cases Plot
pt.figure(figsize=(12, 8))
pt.plot(plot_date, Korea_cases, linestyle='-', linewidth=1.5, color='#000000', label='Korea_cases')
pt.plot(plot_date, Korea_confirm, linestyle='--', linewidth=1.5, color='#000000', label='Korea_Confirmation')
pt.legend(loc=0)
pt.title('Confirmation')
pt.xlabel('Date')
pt.tight_layout()
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/Korea_Confirm.eps",format='eps',dpi=1200, bbox_inches='tight')
pt.show()

print("Elapsed time:", time.time() - start)

In [ ]:
# 서울 & 경기도

from scipy.stats import poisson
from scipy.stats import nbinom
import numpy as np
import matplotlib.pyplot as pt
import time
import pandas as pd
import matplotlib.dates as mdates
from pandas.plotting import register_matplotlib_converters


scale = 1
window = 7
Ntot = 22_689_358.5
dt = 1.0
cases = 1_000_000
sigma = 1.0 / 4.1
beta_s = 0.15  # particle filtering

# xi
df_gamma = pd.read_csv("data/mobility_factor/2016~2017/Seoul_gamma.csv")
xi = df_gamma['gamma']

def moving_average(x, w):
    x = np.asarray(x, dtype=float)
    kernel = np.ones(w)

    # 길이는 그대로 유지
    num = np.convolve(x, kernel, mode='same')
    den = np.convolve(np.ones(len(x)), kernel, mode='same')

    return num / den

# Runge-Kutta 4th order
def RK4(S, I, R, beta, xi):
    kS1 = - xi * beta * S * I / Ntot
    kI1 = (xi * beta * S * I / Ntot) - sigma * I
    kR1 = sigma * I

    kS2 = - xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot
    kI2 = (xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot) - sigma * (I + 0.5 * kI1 * dt)
    kR2 = sigma * (I + 0.5 * kI1 * dt)

    kS3 = - xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot
    kI3 = (xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot) - sigma * (I + 0.5 * kI2 * dt)
    kR3 = sigma * (I + 0.5 * kI2 * dt)

    kS4 = - xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot
    kI4 = (xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot) - sigma * (I + kI3 * dt)
    kR4 = sigma * (I + kI3 * dt)

    S = S + (kS1 + 2.0 * kS2 + 2.0 * kS3 + kS4) / 6.0
    I = I + (kI1 + 2.0 * kI2 + 2.0 * kI3 + kI4) / 6.0
    R = R + (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0
    confirm = (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0

    return np.array([S, I, R, confirm])

def transform_datetype(df):
    df['date'] = pd.to_datetime(df['date'])
    return df

np.random.seed()
start = time.time()

# CSV 읽기
data = pd.read_csv("data/NHIS/2016~2017/Seoul&Gyeonggi_cases(2016~2017).csv")
data = transform_datetype(data)

# cases 열만 숫자로 변환
data['cases'] = pd.to_numeric(data['cases'], errors='coerce')

# 숫자로 변환 안 되는 값 제거
data = data.dropna(subset=['cases']).reset_index(drop=True)

# 원본 cases 배열
cases_array = data['cases'].to_numpy(dtype=float)

# moving average 적용
Seoul_cases = moving_average(cases_array, window)

# Particle filter Rt
day = len(Seoul_cases)

dates = pd.to_datetime(data['date'])
plot_date = dates[0:]

# Initial condition and beta
random_numbers = np.random.normal(0, beta_s, cases * day)
normal_beta = np.reshape(random_numbers, (day, cases))

Pf_S = np.zeros([day + 1, cases])
Pf_I = np.zeros([day + 1, cases])
Pf_R = np.zeros([day + 1, cases])
Pf_beta = np.zeros([day + 1, cases])

rv_index = np.zeros([day + 1, cases], dtype=int)
Pf_confirm = np.zeros([day, cases])
confirm = np.zeros(day)
Pf_RPN = np.zeros(day)
tm = np.zeros(day)

lambda_i = Seoul_cases[0]

Pf_I[0, :] = np.random.poisson(lambda_i, size=cases)
Pf_S[0, :] = Ntot - Pf_I[0, :] - Pf_R[0, :]
Pf_beta[0, :] = 1.05 * np.exp(normal_beta[0, :]) * sigma * Ntot / Pf_S[0, :]

n = 40  # dispersion parameter (Negative Binomial distribution)

# time evolution
for k in range(day):
    # print("Calculation:", k)
    tm[k] = k

    Pf_beta[k + 1, :] = Pf_beta[k, :] * np.exp(normal_beta[k, :])
    Pf_results = RK4(Pf_S[k, :], Pf_I[k, :], Pf_R[k, :], Pf_beta[k + 1, :], xi[k])

    Pf_S[k + 1, :] = Pf_results[0]
    Pf_I[k + 1, :] = Pf_results[1]
    Pf_R[k + 1, :] = Pf_results[2]
    Pf_confirm[k, :] = Pf_results[3]

    # Negative Binomial weight
    weight = nbinom.pmf(round(Seoul_cases[k]), n, n / (Pf_confirm[k, :] + n))

    # weight 합이 0이 되는 경우 방지
    weight_sum = np.sum(weight)
    if weight_sum == 0 or not np.isfinite(weight_sum):
        weight = np.ones(cases) / cases
    else:
        weight = weight / weight_sum

    randomList = np.random.choice(np.arange(cases), size=cases, replace=True, p=weight)
    rv_index[k + 1, :] = randomList

    Pf_S[k + 1, :] = Pf_S[k + 1, randomList]
    Pf_I[k + 1, :] = Pf_I[k + 1, randomList]
    Pf_R[k + 1, :] = Pf_R[k + 1, randomList]
    Pf_beta[k + 1, :] = Pf_beta[k + 1, randomList]

Pf_RPN = xi * (np.mean(Pf_beta[1:, :], axis=1) / sigma) / (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Seoul_confirm = np.mean(Pf_confirm, axis=1)

# Particle smoother Rt
Ps_S = np.zeros([day + 1, cases])
Ps_beta = np.zeros([day + 1, cases])
Ps_RPN = np.zeros(day)

# t = day
Ps_Pf = np.arange(cases)
Ps_beta[day, :] = Pf_beta[day, Ps_Pf]

# t = day - 1
Ps_Pf = rv_index[day, Ps_Pf]
Ps_beta[day - 1, :] = Pf_beta[day - 1, Ps_Pf]

for k in range(day - 2, -1, -1):
    Ps_Pf = rv_index[k + 1, Ps_Pf]
    Ps_beta[k, :] = Pf_beta[k, Ps_Pf]


xi_1 = df_gamma['gamma_1']
xi_2 = df_gamma['gamma_2']
xi_3 = df_gamma['gamma_3']
xi_4 = df_gamma['gamma_4']
xi_5 = df_gamma['gamma_5']

Seoul_Rt = xi * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Seoul_Rt_1 = xi_1 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Seoul_Rt_2 = xi_2 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Seoul_Rt_3 = xi_3 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Seoul_Rt_4 = xi_4 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Seoul_Rt_5 = xi_5 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)

Seoul_df = pd.DataFrame({
    'date': dates[0:],
    'Seoul_Rt': Seoul_Rt,
    'Seoul_Rt_1': Seoul_Rt_1,
    'Seoul_Rt_2': Seoul_Rt_2,
    'Seoul_Rt_3': Seoul_Rt_3,
    'Seoul_Rt_4': Seoul_Rt_4,
    'Seoul_Rt_5': Seoul_Rt_5
})
Seoul_df.to_csv('data/Rt/2016~2017/Seoul_Rt(2016~2017).csv', index=False)

Seoul_beta = np.mean(Ps_beta[1:, :], axis=1)



# Rt Plot
pt.plot(plot_date[10:], Seoul_Rt[10:], linestyle='-', linewidth=1.5, color='#D62728',  label='$R_t$')
pt.title('Seoul Rt')
pt.legend(loc=0)
pt.xlabel('Date')
pt.yticks([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
pt.show()


# Plot
pt.figure(figsize=(12, 8))
pt.plot(plot_date, Seoul_cases, linestyle='-', linewidth=1.5, color='#D62728', label='Seoul_cases')
pt.plot(plot_date, Seoul_confirm, linestyle='--', linewidth=1.5, color='#D62728', label='Seoul_Confirmation')
pt.legend(loc=0)
pt.title('Confirmation')
pt.xlabel('Date')
pt.tight_layout()
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/Seoul_Confirm.eps",format='eps',dpi=1200, bbox_inches='tight')
pt.show()

print("Elapsed time:", time.time() - start)

In [ ]:
# 부산

from scipy.stats import poisson
from scipy.stats import nbinom
import numpy as np
import matplotlib.pyplot as pt
import time
import pandas as pd
import matplotlib.dates as mdates
from pandas.plotting import register_matplotlib_converters


scale = 1
window = 7
Ntot = 3_484_591
dt = 1.0
cases = 1_000_000
sigma = 1.0 / 4.1
beta_s = 0.15  # particle filtering

# xi
df_gamma = pd.read_csv("data/mobility_factor/2016~2017/Busan_gamma.csv")
xi = df_gamma['gamma']

def moving_average(x, w):
    x = np.asarray(x, dtype=float)
    kernel = np.ones(w)

    # 길이는 그대로 유지
    num = np.convolve(x, kernel, mode='same')
    den = np.convolve(np.ones(len(x)), kernel, mode='same')

    return num / den

# Runge-Kutta 4th order
def RK4(S, I, R, beta, xi):
    kS1 = - xi * beta * S * I / Ntot
    kI1 = (xi * beta * S * I / Ntot) - sigma * I
    kR1 = sigma * I

    kS2 = - xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot
    kI2 = (xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot) - sigma * (I + 0.5 * kI1 * dt)
    kR2 = sigma * (I + 0.5 * kI1 * dt)

    kS3 = - xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot
    kI3 = (xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot) - sigma * (I + 0.5 * kI2 * dt)
    kR3 = sigma * (I + 0.5 * kI2 * dt)

    kS4 = - xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot
    kI4 = (xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot) - sigma * (I + kI3 * dt)
    kR4 = sigma * (I + kI3 * dt)

    S = S + (kS1 + 2.0 * kS2 + 2.0 * kS3 + kS4) / 6.0
    I = I + (kI1 + 2.0 * kI2 + 2.0 * kI3 + kI4) / 6.0
    R = R + (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0
    confirm = (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0

    return np.array([S, I, R, confirm])

def transform_datetype(df):
    df['date'] = pd.to_datetime(df['date'])
    return df

np.random.seed()
start = time.time()

# CSV 읽기
data = pd.read_csv("data/NHIS/2016~2017/Busan_cases(2016~2017).csv")
data = transform_datetype(data)

# cases 열만 숫자로 변환
data['cases'] = pd.to_numeric(data['cases'], errors='coerce')

# 숫자로 변환 안 되는 값 제거
data = data.dropna(subset=['cases']).reset_index(drop=True)

# 원본 cases 배열
cases_array = data['cases'].to_numpy(dtype=float)

# moving average 적용
Busan_cases = moving_average(cases_array, window)

# Particle filter Rt
day = len(Busan_cases)

dates = pd.to_datetime(data['date'])
plot_date = dates[0:]

# Initial condition and beta
random_numbers = np.random.normal(0, beta_s, cases * day)
normal_beta = np.reshape(random_numbers, (day, cases))

Pf_S = np.zeros([day + 1, cases])
Pf_I = np.zeros([day + 1, cases])
Pf_R = np.zeros([day + 1, cases])
Pf_beta = np.zeros([day + 1, cases])

rv_index = np.zeros([day + 1, cases], dtype=int)
Pf_confirm = np.zeros([day, cases])
confirm = np.zeros(day)
Pf_RPN = np.zeros(day)
tm = np.zeros(day)

lambda_i = Busan_cases[0]

Pf_I[0, :] = np.random.poisson(lambda_i, size=cases)
Pf_S[0, :] = Ntot - Pf_I[0, :] - Pf_R[0, :]
Pf_beta[0, :] = 1.05 * np.exp(normal_beta[0, :]) * sigma * Ntot / Pf_S[0, :]

n = 40  # dispersion parameter (Negative Binomial distribution)

# time evolution
for k in range(day):
    # print("Calculation:", k)
    tm[k] = k

    Pf_beta[k + 1, :] = Pf_beta[k, :] * np.exp(normal_beta[k, :])
    Pf_results = RK4(Pf_S[k, :], Pf_I[k, :], Pf_R[k, :], Pf_beta[k + 1, :], xi[k])

    Pf_S[k + 1, :] = Pf_results[0]
    Pf_I[k + 1, :] = Pf_results[1]
    Pf_R[k + 1, :] = Pf_results[2]
    Pf_confirm[k, :] = Pf_results[3]

    # Negative Binomial weight
    weight = nbinom.pmf(round(Busan_cases[k]), n, n / (Pf_confirm[k, :] + n))

    # weight 합이 0이 되는 경우 방지
    weight_sum = np.sum(weight)
    if weight_sum == 0 or not np.isfinite(weight_sum):
        weight = np.ones(cases) / cases
    else:
        weight = weight / weight_sum

    randomList = np.random.choice(np.arange(cases), size=cases, replace=True, p=weight)
    rv_index[k + 1, :] = randomList

    Pf_S[k + 1, :] = Pf_S[k + 1, randomList]
    Pf_I[k + 1, :] = Pf_I[k + 1, randomList]
    Pf_R[k + 1, :] = Pf_R[k + 1, randomList]
    Pf_beta[k + 1, :] = Pf_beta[k + 1, randomList]

Pf_RPN = xi * (np.mean(Pf_beta[1:, :], axis=1) / sigma) / (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Busan_confirm = np.mean(Pf_confirm, axis=1)

# Particle smoother Rt
Ps_S = np.zeros([day + 1, cases])
Ps_beta = np.zeros([day + 1, cases])
Ps_RPN = np.zeros(day)

# t = day
Ps_Pf = np.arange(cases)
Ps_beta[day, :] = Pf_beta[day, Ps_Pf]

# t = day - 1
Ps_Pf = rv_index[day, Ps_Pf]
Ps_beta[day - 1, :] = Pf_beta[day - 1, Ps_Pf]

for k in range(day - 2, -1, -1):
    Ps_Pf = rv_index[k + 1, Ps_Pf]
    Ps_beta[k, :] = Pf_beta[k, Ps_Pf]

xi_1 = df_gamma['gamma_1']
xi_2 = df_gamma['gamma_2']
xi_3 = df_gamma['gamma_3']
xi_4 = df_gamma['gamma_4']
xi_5 = df_gamma['gamma_5']

Busan_Rt = xi * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Busan_Rt_1 = xi_1 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Busan_Rt_2 = xi_2 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Busan_Rt_3 = xi_3 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Busan_Rt_4 = xi_4 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Busan_Rt_5 = xi_5 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)

Busan_df = pd.DataFrame({
    'date': dates[0:],  
    'Busan_Rt': Busan_Rt,
    'Busan_Rt_1': Busan_Rt_1,
    'Busan_Rt_2': Busan_Rt_2,
    'Busan_Rt_3': Busan_Rt_3,
    'Busan_Rt_4': Busan_Rt_4,
    'Busan_Rt_5': Busan_Rt_5
})
Busan_df.to_csv('data/Rt/2016~2017/Busan_Rt(2016~2017).csv', index=False)

Busan_beta = np.mean(Ps_beta[1:, :], axis=1)


# Rt Plot
pt.plot(plot_date[10:], Busan_Rt[10:], linestyle='-', linewidth=1.5, color='#1F77B4',  label='$R_t$')
pt.title('Busan Rt')
pt.legend(loc=0)
pt.xlabel('Date')
pt.yticks([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
pt.show()

# Plot
pt.figure(figsize=(12, 8))
pt.plot(plot_date, Busan_cases, linestyle='-', linewidth=1.5, color='#1F77B4', label='Busan_cases')
pt.plot(plot_date, Busan_confirm, linestyle='--', linewidth=1.5, color='#1F77B4', label='Busan_Confirmation')
pt.legend(loc=0)
pt.title('Confirmation')
pt.xlabel('Date')
pt.tight_layout()
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/Busan_Confirm.eps",format='eps',dpi=1200, bbox_inches='tight')
pt.show()

print("Elapsed time:", time.time() - start)

In [ ]:
# 대전

from scipy.stats import poisson
from scipy.stats import nbinom
import numpy as np
import matplotlib.pyplot as pt
import time
import pandas as pd
import matplotlib.dates as mdates
from pandas.plotting import register_matplotlib_converters


scale = 1
window = 7
Ntot = 1_508_298.5
dt = 1.0
cases = 1_000_000
sigma = 1.0 / 4.1
beta_s = 0.15  # particle filtering

# xi
df_gamma = pd.read_csv("data/mobility_factor/2016~2017/Daejeon_gamma.csv")
xi = df_gamma['gamma']

def moving_average(x, w):
    x = np.asarray(x, dtype=float)
    kernel = np.ones(w)

    # 길이는 그대로 유지
    num = np.convolve(x, kernel, mode='same')
    den = np.convolve(np.ones(len(x)), kernel, mode='same')

    return num / den

# Runge-Kutta 4th order
def RK4(S, I, R, beta, xi):
    kS1 = - xi * beta * S * I / Ntot
    kI1 = (xi * beta * S * I / Ntot) - sigma * I
    kR1 = sigma * I

    kS2 = - xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot
    kI2 = (xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot) - sigma * (I + 0.5 * kI1 * dt)
    kR2 = sigma * (I + 0.5 * kI1 * dt)

    kS3 = - xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot
    kI3 = (xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot) - sigma * (I + 0.5 * kI2 * dt)
    kR3 = sigma * (I + 0.5 * kI2 * dt)

    kS4 = - xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot
    kI4 = (xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot) - sigma * (I + kI3 * dt)
    kR4 = sigma * (I + kI3 * dt)

    S = S + (kS1 + 2.0 * kS2 + 2.0 * kS3 + kS4) / 6.0
    I = I + (kI1 + 2.0 * kI2 + 2.0 * kI3 + kI4) / 6.0
    R = R + (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0
    confirm = (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0

    return np.array([S, I, R, confirm])

def transform_datetype(df):
    df['date'] = pd.to_datetime(df['date'])
    return df

np.random.seed()
start = time.time()

# CSV 읽기
data = pd.read_csv("data/NHIS/2016~2017/Daejeon_cases(2016~2017).csv")
data = transform_datetype(data)

# cases 열만 숫자로 변환
data['cases'] = pd.to_numeric(data['cases'], errors='coerce')

# 숫자로 변환 안 되는 값 제거
data = data.dropna(subset=['cases']).reset_index(drop=True)

# 원본 cases 배열
cases_array = data['cases'].to_numpy(dtype=float)

# moving average 적용
Daejeon_cases = moving_average(cases_array, window)

# Particle filter Rt
day = len(Daejeon_cases)

dates = pd.to_datetime(data['date'])
plot_date = dates[0:]

# Initial condition and beta
random_numbers = np.random.normal(0, beta_s, cases * day)
normal_beta = np.reshape(random_numbers, (day, cases))

Pf_S = np.zeros([day + 1, cases])
Pf_I = np.zeros([day + 1, cases])
Pf_R = np.zeros([day + 1, cases])
Pf_beta = np.zeros([day + 1, cases])

rv_index = np.zeros([day + 1, cases], dtype=int)
Pf_confirm = np.zeros([day, cases])
confirm = np.zeros(day)
Pf_RPN = np.zeros(day)
tm = np.zeros(day)

lambda_i = Daejeon_cases[0]

Pf_I[0, :] = np.random.poisson(lambda_i, size=cases)
Pf_S[0, :] = Ntot - Pf_I[0, :] - Pf_R[0, :]
Pf_beta[0, :] = 1.05 * np.exp(normal_beta[0, :]) * sigma * Ntot / Pf_S[0, :]

n = 40  # dispersion parameter (Negative Binomial distribution)

# time evolution
for k in range(day):
    # print("Calculation:", k)
    tm[k] = k

    Pf_beta[k + 1, :] = Pf_beta[k, :] * np.exp(normal_beta[k, :])
    Pf_results = RK4(Pf_S[k, :], Pf_I[k, :], Pf_R[k, :], Pf_beta[k + 1, :], xi[k])

    Pf_S[k + 1, :] = Pf_results[0]
    Pf_I[k + 1, :] = Pf_results[1]
    Pf_R[k + 1, :] = Pf_results[2]
    Pf_confirm[k, :] = Pf_results[3]

    # Negative Binomial weight
    weight = nbinom.pmf(round(Daejeon_cases[k]), n, n / (Pf_confirm[k, :] + n))

    # weight 합이 0이 되는 경우 방지
    weight_sum = np.sum(weight)
    if weight_sum == 0 or not np.isfinite(weight_sum):
        weight = np.ones(cases) / cases
    else:
        weight = weight / weight_sum

    randomList = np.random.choice(np.arange(cases), size=cases, replace=True, p=weight)
    rv_index[k + 1, :] = randomList

    Pf_S[k + 1, :] = Pf_S[k + 1, randomList]
    Pf_I[k + 1, :] = Pf_I[k + 1, randomList]
    Pf_R[k + 1, :] = Pf_R[k + 1, randomList]
    Pf_beta[k + 1, :] = Pf_beta[k + 1, randomList]

Pf_RPN = xi * (np.mean(Pf_beta[1:, :], axis=1) / sigma) / (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daejeon_confirm = np.mean(Pf_confirm, axis=1)

# Particle smoother Rt
Ps_S = np.zeros([day + 1, cases])
Ps_beta = np.zeros([day + 1, cases])
Ps_RPN = np.zeros(day)

# t = day
Ps_Pf = np.arange(cases)
Ps_beta[day, :] = Pf_beta[day, Ps_Pf]

# t = day - 1
Ps_Pf = rv_index[day, Ps_Pf]
Ps_beta[day - 1, :] = Pf_beta[day - 1, Ps_Pf]

for k in range(day - 2, -1, -1):
    Ps_Pf = rv_index[k + 1, Ps_Pf]
    Ps_beta[k, :] = Pf_beta[k, Ps_Pf]

xi_1 = df_gamma['gamma_1']
xi_2 = df_gamma['gamma_2']
xi_3 = df_gamma['gamma_3']
xi_4 = df_gamma['gamma_4']
xi_5 = df_gamma['gamma_5']

Daejeon_Rt = xi * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daejeon_Rt_1 = xi_1 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daejeon_Rt_2 = xi_2 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daejeon_Rt_3 = xi_3 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daejeon_Rt_4 = xi_4 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daejeon_Rt_5 = xi_5 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)

Daejeon_df = pd.DataFrame({
    'date': dates[0:],  
    'Daejeon_Rt': Daejeon_Rt,
    'Daejeon_Rt_1': Daejeon_Rt_1,
    'Daejeon_Rt_2': Daejeon_Rt_2,
    'Daejeon_Rt_3': Daejeon_Rt_3,
    'Daejeon_Rt_4': Daejeon_Rt_4,
    'Daejeon_Rt_5': Daejeon_Rt_5
})
Daejeon_df.to_csv('data/Rt/2016~2017/Daejeon_Rt(2016~2017).csv', index=False)

Daejeon_beta = np.mean(Ps_beta[1:, :], axis=1)

# Rt Plot
pt.plot(plot_date[10:], Daejeon_Rt[10:], linestyle='-', linewidth=1.5, color='#2CA02C',  label='$R_t$')
pt.title('Daejeon Rt')
pt.legend(loc=0)
pt.xlabel('Date')
pt.yticks([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
pt.show()

# Plot
pt.figure(figsize=(12, 8))
pt.plot(plot_date, Daejeon_cases, linestyle='-', linewidth=1.5, color='#2CA02C', label='Daejeon_cases')
pt.plot(plot_date, Daejeon_confirm, linestyle='--', linewidth=1.5, color='#2CA02C', label='Daejeon_Confirmation')
pt.legend(loc=0)
pt.title('Confirmation')
pt.xlabel('Date')
pt.tight_layout()
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/Daejeon_Confirm.eps",format='eps',dpi=1200, bbox_inches='tight')
pt.show()

print("Elapsed time:", time.time() - start)

In [ ]:
# 대구

from scipy.stats import poisson
from scipy.stats import nbinom
import numpy as np
import matplotlib.pyplot as pt
import time
import pandas as pd
import matplotlib.dates as mdates
from pandas.plotting import register_matplotlib_converters


scale = 1
window = 7
Ntot = 2_479_894
dt = 1.0
cases = 1_000_000
sigma = 1.0 / 4.1
beta_s = 0.15  # particle filtering

# xi
df_gamma = pd.read_csv("data/mobility_factor/2016~2017/Daegu_gamma.csv")
xi = df_gamma['gamma']

def moving_average(x, w):
    x = np.asarray(x, dtype=float)
    kernel = np.ones(w)

    # 길이는 그대로 유지
    num = np.convolve(x, kernel, mode='same')
    den = np.convolve(np.ones(len(x)), kernel, mode='same')

    return num / den

# Runge-Kutta 4th order
def RK4(S, I, R, beta, xi):
    kS1 = - xi * beta * S * I / Ntot
    kI1 = (xi * beta * S * I / Ntot) - sigma * I
    kR1 = sigma * I

    kS2 = - xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot
    kI2 = (xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot) - sigma * (I + 0.5 * kI1 * dt)
    kR2 = sigma * (I + 0.5 * kI1 * dt)

    kS3 = - xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot
    kI3 = (xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot) - sigma * (I + 0.5 * kI2 * dt)
    kR3 = sigma * (I + 0.5 * kI2 * dt)

    kS4 = - xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot
    kI4 = (xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot) - sigma * (I + kI3 * dt)
    kR4 = sigma * (I + kI3 * dt)

    S = S + (kS1 + 2.0 * kS2 + 2.0 * kS3 + kS4) / 6.0
    I = I + (kI1 + 2.0 * kI2 + 2.0 * kI3 + kI4) / 6.0
    R = R + (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0
    confirm = (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0

    return np.array([S, I, R, confirm])

def transform_datetype(df):
    df['date'] = pd.to_datetime(df['date'])
    return df

np.random.seed()
start = time.time()

# CSV 읽기
data = pd.read_csv("data/NHIS/2016~2017/Daegu_cases(2016~2017).csv")
data = transform_datetype(data)

# cases 열만 숫자로 변환
data['cases'] = pd.to_numeric(data['cases'], errors='coerce')

# 숫자로 변환 안 되는 값 제거
data = data.dropna(subset=['cases']).reset_index(drop=True)

# 원본 cases 배열
cases_array = data['cases'].to_numpy(dtype=float)

# moving average 적용
Daegu_cases = moving_average(cases_array, window)

# Particle filter Rt
day = len(Daegu_cases)

dates = pd.to_datetime(data['date'])
plot_date = dates[0:]

# Initial condition and beta
random_numbers = np.random.normal(0, beta_s, cases * day)
normal_beta = np.reshape(random_numbers, (day, cases))

Pf_S = np.zeros([day + 1, cases])
Pf_I = np.zeros([day + 1, cases])
Pf_R = np.zeros([day + 1, cases])
Pf_beta = np.zeros([day + 1, cases])

rv_index = np.zeros([day + 1, cases], dtype=int)
Pf_confirm = np.zeros([day, cases])
confirm = np.zeros(day)
Pf_RPN = np.zeros(day)
tm = np.zeros(day)

lambda_i = Daegu_cases[0]

Pf_I[0, :] = np.random.poisson(lambda_i, size=cases)
Pf_S[0, :] = Ntot - Pf_I[0, :] - Pf_R[0, :]
Pf_beta[0, :] = 1.05 * np.exp(normal_beta[0, :]) * sigma * Ntot / Pf_S[0, :]

n = 40  # dispersion parameter (Negative Binomial distribution)

# time evolution
for k in range(day):
    # print("Calculation:", k)
    tm[k] = k

    Pf_beta[k + 1, :] = Pf_beta[k, :] * np.exp(normal_beta[k, :])
    Pf_results = RK4(Pf_S[k, :], Pf_I[k, :], Pf_R[k, :], Pf_beta[k + 1, :], xi[k])

    Pf_S[k + 1, :] = Pf_results[0]
    Pf_I[k + 1, :] = Pf_results[1]
    Pf_R[k + 1, :] = Pf_results[2]
    Pf_confirm[k, :] = Pf_results[3]

    # Negative Binomial weight
    weight = nbinom.pmf(round(Daegu_cases[k]), n, n / (Pf_confirm[k, :] + n))

    # weight 합이 0이 되는 경우 방지
    weight_sum = np.sum(weight)
    if weight_sum == 0 or not np.isfinite(weight_sum):
        weight = np.ones(cases) / cases
    else:
        weight = weight / weight_sum

    randomList = np.random.choice(np.arange(cases), size=cases, replace=True, p=weight)
    rv_index[k + 1, :] = randomList

    Pf_S[k + 1, :] = Pf_S[k + 1, randomList]
    Pf_I[k + 1, :] = Pf_I[k + 1, randomList]
    Pf_R[k + 1, :] = Pf_R[k + 1, randomList]
    Pf_beta[k + 1, :] = Pf_beta[k + 1, randomList]

Pf_RPN = xi * (np.mean(Pf_beta[1:, :], axis=1) / sigma) / (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daegu_confirm = np.mean(Pf_confirm, axis=1)

# Particle smoother Rt
Ps_S = np.zeros([day + 1, cases])
Ps_beta = np.zeros([day + 1, cases])
Ps_RPN = np.zeros(day)

# t = day
Ps_Pf = np.arange(cases)
Ps_beta[day, :] = Pf_beta[day, Ps_Pf]

# t = day - 1
Ps_Pf = rv_index[day, Ps_Pf]
Ps_beta[day - 1, :] = Pf_beta[day - 1, Ps_Pf]

for k in range(day - 2, -1, -1):
    Ps_Pf = rv_index[k + 1, Ps_Pf]
    Ps_beta[k, :] = Pf_beta[k, Ps_Pf]

xi_1 = df_gamma['gamma_1']
xi_2 = df_gamma['gamma_2']
xi_3 = df_gamma['gamma_3']
xi_4 = df_gamma['gamma_4']
xi_5 = df_gamma['gamma_5']

Daegu_Rt = xi * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daegu_Rt_1 = xi_1 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daegu_Rt_2 = xi_2 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daegu_Rt_3 = xi_3 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daegu_Rt_4 = xi_4 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Daegu_Rt_5 = xi_5 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)

Daegu_df = pd.DataFrame({
    'date': dates[0:],  
    'Daegu_Rt': Daegu_Rt,
    'Daegu_Rt_1': Daegu_Rt_1,
    'Daegu_Rt_2': Daegu_Rt_2,
    'Daegu_Rt_3': Daegu_Rt_3,
    'Daegu_Rt_4': Daegu_Rt_4,
    'Daegu_Rt_5': Daegu_Rt_5
})
Daegu_df.to_csv('data/Rt/2016~2017/Daegu_Rt(2016~2017).csv', index=False)

Daegu_beta = np.mean(Ps_beta[1:, :], axis=1)

# Rt Plot
pt.plot(plot_date[10:], Daegu_Rt[10:], linestyle='-', linewidth=1.5, color='#FF7F0E',  label='$R_t$')
pt.title('Daegu Rt')
pt.legend(loc=0)
pt.xlabel('Date')
pt.yticks([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
pt.show()

# Plot
pt.figure(figsize=(12, 8))
pt.plot(plot_date, Daegu_cases, linestyle='-', linewidth=1.5, color='#FF7F0E', label='Daegu_cases')
pt.plot(plot_date, Daegu_confirm, linestyle='--', linewidth=1.5, color='#FF7F0E', label='Daegu_Confirmation')
pt.legend(loc=0)
pt.title('Confirmation')
pt.xlabel('Date')
pt.tight_layout()
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/Daegu_Confirm.eps",format='eps',dpi=1200, bbox_inches='tight')
pt.show()

print("Elapsed time:", time.time() - start)

In [ ]:
# 광주

from scipy.stats import poisson
from scipy.stats import nbinom
import numpy as np
import matplotlib.pyplot as pt
import time
import pandas as pd
import matplotlib.dates as mdates
from pandas.plotting import register_matplotlib_converters


scale = 1
window = 7
Ntot = 1_466_492
dt = 1.0
cases = 1_000_000
sigma = 1.0 / 4.1
beta_s = 0.15  # particle filtering

# xi
df_gamma = pd.read_csv("data/mobility_factor/2016~2017/Gwangju_gamma.csv")
xi = df_gamma['gamma']

def moving_average(x, w):
    x = np.asarray(x, dtype=float)
    kernel = np.ones(w)

    # 길이는 그대로 유지
    num = np.convolve(x, kernel, mode='same')
    den = np.convolve(np.ones(len(x)), kernel, mode='same')

    return num / den

# Runge-Kutta 4th order
def RK4(S, I, R, beta, xi):
    kS1 = - xi * beta * S * I / Ntot
    kI1 = (xi * beta * S * I / Ntot) - sigma * I
    kR1 = sigma * I

    kS2 = - xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot
    kI2 = (xi * beta * (S + 0.5 * kS1 * dt) * (I + 0.5 * kI1 * dt) / Ntot) - sigma * (I + 0.5 * kI1 * dt)
    kR2 = sigma * (I + 0.5 * kI1 * dt)

    kS3 = - xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot
    kI3 = (xi * beta * (S + 0.5 * kS2 * dt) * (I + 0.5 * kI2 * dt) / Ntot) - sigma * (I + 0.5 * kI2 * dt)
    kR3 = sigma * (I + 0.5 * kI2 * dt)

    kS4 = - xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot
    kI4 = (xi * beta * (S + kS3 * dt) * (I + kI3 * dt) / Ntot) - sigma * (I + kI3 * dt)
    kR4 = sigma * (I + kI3 * dt)

    S = S + (kS1 + 2.0 * kS2 + 2.0 * kS3 + kS4) / 6.0
    I = I + (kI1 + 2.0 * kI2 + 2.0 * kI3 + kI4) / 6.0
    R = R + (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0
    confirm = (kR1 + 2.0 * kR2 + 2.0 * kR3 + kR4) / 6.0

    return np.array([S, I, R, confirm])

def transform_datetype(df):
    df['date'] = pd.to_datetime(df['date'])
    return df

np.random.seed()
start = time.time()

# CSV 읽기
data = pd.read_csv('data/NHIS/2016~2017/Gwangju_cases(2016~2017).csv')
data = transform_datetype(data)

# cases 열만 숫자로 변환
data['cases'] = pd.to_numeric(data['cases'], errors='coerce')

# 숫자로 변환 안 되는 값 제거
data = data.dropna(subset=['cases']).reset_index(drop=True)

# 원본 cases 배열
cases_array = data['cases'].to_numpy(dtype=float)

# moving average 적용
gwangju_cases = moving_average(cases_array, window)

# Particle filter Rt
day = len(gwangju_cases)

dates = pd.to_datetime(data['date'])
plot_date = dates[0:]

# Initial condition and beta
random_numbers = np.random.normal(0, beta_s, cases * day)
normal_beta = np.reshape(random_numbers, (day, cases))

Pf_S = np.zeros([day + 1, cases])
Pf_I = np.zeros([day + 1, cases])
Pf_R = np.zeros([day + 1, cases])
Pf_beta = np.zeros([day + 1, cases])

rv_index = np.zeros([day + 1, cases], dtype=int)
Pf_confirm = np.zeros([day, cases])
confirm = np.zeros(day)
Pf_RPN = np.zeros(day)
tm = np.zeros(day)

lambda_i = gwangju_cases[0]

Pf_I[0, :] = np.random.poisson(lambda_i, size=cases)
Pf_S[0, :] = Ntot - Pf_I[0, :] - Pf_R[0, :]
Pf_beta[0, :] = 1.05 * np.exp(normal_beta[0, :]) * sigma * Ntot / Pf_S[0, :]

n = 40  # dispersion parameter (Negative Binomial distribution)

# time evolution
for k in range(day):
    # print("Calculation:", k)
    tm[k] = k

    Pf_beta[k + 1, :] = Pf_beta[k, :] * np.exp(normal_beta[k, :])
    Pf_results = RK4(Pf_S[k, :], Pf_I[k, :], Pf_R[k, :], Pf_beta[k + 1, :], xi[k])

    Pf_S[k + 1, :] = Pf_results[0]
    Pf_I[k + 1, :] = Pf_results[1]
    Pf_R[k + 1, :] = Pf_results[2]
    Pf_confirm[k, :] = Pf_results[3]

    # Negative Binomial weight
    weight = nbinom.pmf(round(gwangju_cases[k]), n, n / (Pf_confirm[k, :] + n))

    # weight 합이 0이 되는 경우 방지
    weight_sum = np.sum(weight)
    if weight_sum == 0 or not np.isfinite(weight_sum):
        weight = np.ones(cases) / cases
    else:
        weight = weight / weight_sum

    randomList = np.random.choice(np.arange(cases), size=cases, replace=True, p=weight)
    rv_index[k + 1, :] = randomList

    Pf_S[k + 1, :] = Pf_S[k + 1, randomList]
    Pf_I[k + 1, :] = Pf_I[k + 1, randomList]
    Pf_R[k + 1, :] = Pf_R[k + 1, randomList]
    Pf_beta[k + 1, :] = Pf_beta[k + 1, randomList]

Pf_RPN = xi * (np.mean(Pf_beta[1:, :], axis=1) / sigma) / (np.mean(Pf_S[1:, :], axis=1) / Ntot)
gwangju_confirm = np.mean(Pf_confirm, axis=1)

# Particle smoother Rt
Ps_S = np.zeros([day + 1, cases])
Ps_beta = np.zeros([day + 1, cases])
Ps_RPN = np.zeros(day)

# t = day
Ps_Pf = np.arange(cases)
Ps_beta[day, :] = Pf_beta[day, Ps_Pf]

# t = day - 1
Ps_Pf = rv_index[day, Ps_Pf]
Ps_beta[day - 1, :] = Pf_beta[day - 1, Ps_Pf]

for k in range(day - 2, -1, -1):
    Ps_Pf = rv_index[k + 1, Ps_Pf]
    Ps_beta[k, :] = Pf_beta[k, Ps_Pf]

xi_1 = df_gamma['gamma_1']
xi_2 = df_gamma['gamma_2']
xi_3 = df_gamma['gamma_3']
xi_4 = df_gamma['gamma_4']
xi_5 = df_gamma['gamma_5']

Gwangju_Rt = xi * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Gwangju_Rt_1 = xi_1 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Gwangju_Rt_2 = xi_2 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Gwangju_Rt_3 = xi_3 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Gwangju_Rt_4 = xi_4 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)
Gwangju_Rt_5 = xi_5 * (np.mean(Ps_beta[1:, :], axis=1) / sigma) * (np.mean(Pf_S[1:, :], axis=1) / Ntot)

Gwangju_df = pd.DataFrame({
    'date': dates[0:],  
    'Gwangju_Rt': Gwangju_Rt,
    'Gwangju_Rt_1': Gwangju_Rt_1,
    'Gwangju_Rt_2': Gwangju_Rt_2,
    'Gwangju_Rt_3': Gwangju_Rt_3,
    'Gwangju_Rt_4': Gwangju_Rt_4,
    'Gwangju_Rt_5': Gwangju_Rt_5
})
Gwangju_df.to_csv('data/Rt/2016~2017/Gwangju_Rt(2016~2017).csv', index=False)

Gwangju_beta = np.mean(Ps_beta[1:, :], axis=1)

# Rt Plot
pt.plot(plot_date[10:], Gwangju_Rt[10:], linestyle='-', linewidth=1.5, color='#9467BD',  label='$R_t$')
pt.title('Gwangju Rt')
pt.legend(loc=0)
pt.xlabel('Date')
pt.yticks([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
pt.show()

# Plot
pt.figure(figsize=(18, 8))
pt.plot(plot_date, gwangju_cases, linestyle='-', linewidth=1.5, color='#9467BD', label='Gwangju_cases')
pt.plot(plot_date, gwangju_confirm, linestyle='--', linewidth=1.5, color='#9467BD', label='Gwangju_Confirmation')
pt.legend(loc=0)
pt.title('Gwangju_Confirmation')
pt.xlabel('Date')
pt.tight_layout()
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/Gwangju_Confirm.eps",format='eps',dpi=1200, bbox_inches='tight')
pt.show()

print("Elapsed time:", time.time() - start)

In [ ]:
# plot
pt.figure(figsize=(18, 8))
pt.plot(plot_date, Seoul_cases, linestyle='-', linewidth=1.5, color='#D62728', label='Seoul_cases')
pt.plot(plot_date, Seoul_confirm, linestyle='--', linewidth=1.5, color='#D62728', label='Seoul_Confirm')
pt.plot(plot_date, Busan_cases, linestyle='-', linewidth=1.5, color='#1F77B4', label='Busan_cases')
pt.plot(plot_date, Busan_confirm, linestyle='--', linewidth=1.5, color='#1F77B4', label='Busan_Confirm')
pt.plot(plot_date, Daejeon_cases, linestyle='-', linewidth=1.5, color='#2CA02C', label='Daejeon_cases')
pt.plot(plot_date, Daejeon_confirm, linestyle='--', linewidth=1.5, color='#2CA02C', label='Daejeon_Confirm')
pt.plot(plot_date, Daegu_cases, linestyle='-', linewidth=1.5, color='#FF7F0E', label='Daegu_cases')
pt.plot(plot_date, Daegu_confirm, linestyle='--', linewidth=1.5, color='#FF7F0E', label='Daegu_Confirm')
pt.plot(plot_date, gwangju_cases, linestyle='-', linewidth=1.5, color='#9467BD', label='Gwangju_cases')
pt.plot(plot_date, gwangju_confirm, linestyle='--', linewidth=1.5, color='#9467BD', label='Gwangju_Confirm')
pt.legend(loc=0)
pt.title('Regional_Confirmation')
pt.xlabel('Date')
pt.tight_layout()
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/2016~2017_confirm.eps",format='eps',dpi=1200, bbox_inches='tight')
# pt.savefig('2016~2017_confirm.png', format='png', bbox_inches='tight')
pt.show()

In [ ]:
import matplotlib.pyplot as plt
colors = ['#000000', '#D62728', '#1F77B4', '#2CA02C', '#FF7F0E', '#9467BD', '#8C564B']

plot_dates = dates[10:]
pt.figure(figsize=(18, 8))
# plt.plot(plot_dates,Korea_beta[10:], linestyle='--', linewidth=1.5, color=colors[0], label='Korea')
plt.plot(plot_dates,Seoul_beta[10:], '-', linewidth=1, color=colors[1],  label='Seoul')
plt.plot(plot_dates,Busan_beta[10:], '-', linewidth=1, color=colors[2], label='Busan')
plt.plot(plot_dates,Daejeon_beta[10:], '-', linewidth=1, color=colors[3], label='Daejeon')
plt.plot(plot_dates,Daegu_beta[10:], '-', linewidth=1, color=colors[4], label='Daegu')
plt.plot(plot_dates,Gwangju_beta[10:], '-', linewidth=1, color=colors[5], label='Gwangju')
plt.title('Regional beta')
plt.yticks([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])
plt.legend(loc=0)
plt.ylabel('beta')
plt.xlabel('Date')
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/2016~2017_beta.eps",format='eps',dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
colors = ['#000000', '#D62728', '#1F77B4', '#2CA02C', '#FF7F0E', '#9467BD', '#8C564B']

plot_dates = dates[10:]
pt.figure(figsize=(18, 8))
# plt.plot(plot_dates,Korea_Rt[10:], linestyle='--', linewidth=1.5, color=colors[0], label='Korea')
plt.plot(plot_dates,Seoul_Rt[10:], '-', linewidth=1, color=colors[1],  label='Seoul')
plt.plot(plot_dates,Busan_Rt[10:], '-', linewidth=1, color=colors[2], label='Busan')
plt.plot(plot_dates,Daejeon_Rt[10:], '-', linewidth=1, color=colors[3], label='Daejeon')
plt.plot(plot_dates,Daegu_Rt[10:], '-', linewidth=1, color=colors[4], label='Daegu')
plt.plot(plot_dates,Gwangju_Rt[10:], '-', linewidth=1, color=colors[5], label='Gwangju')
plt.title('Regional $R_t$ Comparison')
plt.legend(loc=0)
plt.yticks([0.4, 0.8, 1.2, 1.6, 2.0, 2.4])
plt.axhline(1.0, linestyle="--", label="Reference ($R_t$ = 1)")
plt.ylabel('$R_t$')
plt.xlabel('Date')
pt.grid(True,which='major',linestyle='-',linewidth=0.5,color='0.85')
pt.savefig("figures/2016~2017/2016~2017_Rt.eps",format='eps',dpi=1200, bbox_inches='tight')
# plt.savefig('2016~2017_0.5_Rt.png', format='png', bbox_inches='tight')
plt.show()